# ConsultaAI — Etapa 1.2: Instanciação, Inferência e Benchmark de Modelos

Este notebook carrega o dataset fatiado na Etapa 1.1, instancia os modelos de ASR (Wav2Vec2, Meta MMS e Whisper), executa a inferência e calcula as métricas de acurácia (WER/CER) em relação aos Gabaritos A (coloquial) e B (normalizado), além de medir tempos de execução.

In [ ]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Adiciona a pasta src ao sys.path para importar os módulos locais
sys.path.append(str(Path("..").resolve()))

from src.models.inference import ASRModelWrapper
from src.evaluation.metrics import calculate_wer, calculate_cer

## 1. Carregamento dos Metadados Preprocessados

Carregamos o arquivo `metadata.csv` contendo a localização dos áudios fatiados e as referências textuais.

In [ ]:
metadata_path = Path("../data/input/processed/metadata.csv")
df = pd.read_csv(metadata_path)
print(f"Total de segmentos disponíveis para benchmark: {len(df)}")
df.head()

## 2. Configuração do Modo de Execução (Teste vs. Definitivo)

Configura se o benchmark deve rodar no modo de teste rápido (100 segmentos) ou no modo definitivo (todos os segmentos do dataset).

In [ ]:
# Controle de Execução:
# - True: Executa o benchmark definitivo em todo o dataset (16.963 segmentos, ~9.5h de áudio).
# - False: Executa o benchmark de teste em 100 segmentos do arquivo 'besqau01'.
USE_FULL_DATASET = True

if USE_FULL_DATASET:
    df_benchmark = df.copy()
    output_filename = 'benchmark_predictions.csv'
    print(f'Modo Definitivo ativo: Avaliando TODOS os {len(df_benchmark)} segmentos.')
else:
    df_benchmark = df[df['file_id'] == 'besqau01'].head(100).copy()
    output_filename = 'benchmark_predictions_100_segments.csv'
    print(f'Modo Teste ativo: Avaliando amostra com {len(df_benchmark)} segmentos.')

## 3. Avaliação do Modelo Whisper (base)

Instanciamos o Whisper Base via faster-whisper e rodamos a inferência na nossa amostra.

In [ ]:
# Instancia o modelo Whisper Base via faster-whisper
whisper_base_model = ASRModelWrapper(model_type="whisper", model_id_or_size="base")

In [ ]:
whisper_base_preds = []
whisper_base_times = []
audio_durations = []

for _, row in tqdm(df_benchmark.iterrows(), total=len(df_benchmark), desc="Whisper Base"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = whisper_base_model.transcribe(audio_path)
        whisper_base_preds.append(res["text"])
        whisper_base_times.append(res["execution_time_s"])
        audio_durations.append(res["audio_len_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        whisper_base_preds.append("")
        whisper_base_times.append(0.0)
        audio_durations.append(0.0)

df_benchmark["pred_whisper_base"] = whisper_base_preds
df_benchmark["time_whisper_base"] = whisper_base_times
df_benchmark["audio_len"] = audio_durations

## 4. Avaliação do Modelo Whisper (small)

Instanciamos o Whisper Small via faster-whisper e rodamos a inferência na nossa amostra.

In [ ]:
# Instancia o modelo Whisper Small via faster-whisper
whisper_small_model = ASRModelWrapper(model_type="whisper", model_id_or_size="small")

In [ ]:
whisper_small_preds = []
whisper_small_times = []

for _, row in tqdm(df_benchmark.iterrows(), total=len(df_benchmark), desc="Whisper Small"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = whisper_small_model.transcribe(audio_path)
        whisper_small_preds.append(res["text"])
        whisper_small_times.append(res["execution_time_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        whisper_small_preds.append("")
        whisper_small_times.append(0.0)

df_benchmark["pred_whisper_small"] = whisper_small_preds
df_benchmark["time_whisper_small"] = whisper_small_times

## 5. Avaliação do Modelo Wav2Vec2 (Português)

Instanciamos o Wav2Vec2 treinado para o português (`jonatasgrosman/wav2vec2-large-xlsr-53-portuguese`) via Hugging Face e rodamos a inferência.

In [ ]:
# Instancia o modelo Wav2Vec2
wav2vec2_model = ASRModelWrapper(
    model_type="wav2vec2", 
    model_id_or_size="jonatasgrosman/wav2vec2-large-xlsr-53-portuguese"
)

In [ ]:
wav2vec2_preds = []
wav2vec2_times = []

for _, row in tqdm(df_benchmark.iterrows(), total=len(df_benchmark), desc="Wav2Vec2 PT"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = wav2vec2_model.transcribe(audio_path)
        wav2vec2_preds.append(res["text"])
        wav2vec2_times.append(res["execution_time_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        wav2vec2_preds.append("")
        wav2vec2_times.append(0.0)

df_benchmark["pred_wav2vec2_pt"] = wav2vec2_preds
df_benchmark["time_wav2vec2_pt"] = wav2vec2_times

## 6. Avaliação do Modelo Meta MMS-1B (Português)

Instanciamos o Meta MMS-1B (`facebook/mms-1b-all`) via Hugging Face e rodamos a inferência.

In [ ]:
# Instancia o modelo Meta MMS-1B
mms_model = ASRModelWrapper(
    model_type="mms", 
    model_id_or_size="facebook/mms-1b-all"
)

In [ ]:
mms_preds = []
mms_times = []

for _, row in tqdm(df_benchmark.iterrows(), total=len(df_benchmark), desc="Meta MMS-1B"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = mms_model.transcribe(audio_path)
        mms_preds.append(res["text"])
        mms_times.append(res["execution_time_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        mms_preds.append("")
        mms_times.append(0.0)

df_benchmark["pred_mms_1b"] = mms_preds
df_benchmark["time_mms_1b"] = mms_times

## 7. Cálculo de Métricas Comparativas (WER/CER)

Calculamos as taxas de erro das predições de cada modelo comparando contra as referências dos Gabaritos A (coloquial) e B (normalizado).

In [ ]:
# Filtra linhas válidas (onde a inferência funcionou)
df_valid = df_benchmark[df_benchmark["audio_len"] > 0].copy()

# Referências
ref_a = df_valid["text_gabarito_a"].fillna("").tolist()
ref_b = df_valid["text_gabarito_b"].fillna("").tolist()

# Predições
pred_whisper_base = df_valid["pred_whisper_base"].fillna("").tolist()
pred_whisper_small = df_valid["pred_whisper_small"].fillna("").tolist()
pred_wav2vec2 = df_valid["pred_wav2vec2_pt"].fillna("").tolist()
pred_mms = df_valid["pred_mms_1b"].fillna("").tolist()

# WER
wer_whisper_base_a = calculate_wer(ref_a, pred_whisper_base)
wer_whisper_base_b = calculate_wer(ref_b, pred_whisper_base)
wer_whisper_small_a = calculate_wer(ref_a, pred_whisper_small)
wer_whisper_small_b = calculate_wer(ref_b, pred_whisper_small)
wer_wav2vec2_a = calculate_wer(ref_a, pred_wav2vec2)
wer_wav2vec2_b = calculate_wer(ref_b, pred_wav2vec2)
wer_mms_a = calculate_wer(ref_a, pred_mms)
wer_mms_b = calculate_wer(ref_b, pred_mms)

# CER
cer_whisper_base_a = calculate_cer(ref_a, pred_whisper_base)
cer_whisper_base_b = calculate_cer(ref_b, pred_whisper_base)
cer_whisper_small_a = calculate_cer(ref_a, pred_whisper_small)
cer_whisper_small_b = calculate_cer(ref_b, pred_whisper_small)
cer_wav2vec2_a = calculate_cer(ref_a, pred_wav2vec2)
cer_wav2vec2_b = calculate_cer(ref_b, pred_wav2vec2)
cer_mms_a = calculate_cer(ref_a, pred_mms)
cer_mms_b = calculate_cer(ref_b, pred_mms)

# Métricas de Hardware
total_audio_s = df_valid["audio_len"].sum()
rtf_whisper_base = df_valid["time_whisper_base"].sum() / total_audio_s
rtf_whisper_small = df_valid["time_whisper_small"].sum() / total_audio_s
rtf_wav2vec2 = df_valid["time_wav2vec2_pt"].sum() / total_audio_s
rtf_mms = df_valid["time_mms_1b"].sum() / total_audio_s

# Montando o Report
results = {
    "Modelo": ["Whisper Base", "Whisper Small", "Wav2Vec2 PT (XLSR-53)", "Meta MMS-1B"],
    "WER - Gabarito A (Coloquial)": [wer_whisper_base_a, wer_whisper_small_a, wer_wav2vec2_a, wer_mms_a],
    "WER - Gabarito B (Normalizado)": [wer_whisper_base_b, wer_whisper_small_b, wer_wav2vec2_b, wer_mms_b],
    "CER - Gabarito A (Coloquial)": [cer_whisper_base_a, cer_whisper_small_a, cer_wav2vec2_a, cer_mms_a],
    "CER - Gabarito B (Normalizado)": [cer_whisper_base_b, cer_whisper_small_b, cer_wav2vec2_b, cer_mms_b],
    "RTF (Fator de Tempo Real)": [rtf_whisper_base, rtf_whisper_small, rtf_wav2vec2, rtf_mms],
    "Tempo Médio por Segmento (s)": [
        df_valid["time_whisper_base"].mean(),
        df_valid["time_whisper_small"].mean(),
        df_valid["time_wav2vec2_pt"].mean(),
        df_valid["time_mms_1b"].mean()
    ]
}

df_results = pd.DataFrame(results)
df_results

## 8. Análise de Amostras Lado a Lado

Vamos visualizar algumas predições reais comparadas com as referências originais para entender as características e tipos de erros de cada modelo.

In [ ]:
columns_to_show = [
    "speaker",
    "text_gabarito_a",
    "text_gabarito_b",
    "pred_whisper_base",
    "pred_whisper_small",
    "pred_wav2vec2_pt",
    "pred_mms_1b"
]
df_valid[columns_to_show].head(10)

## 9. Salvamento do Dataset de Saída Comparativo

Salvamos as predições completas da amostra em formato CSV para visualização dos resultados


In [ ]:
output_csv_path = Path("../data/output") / output_filename
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
df_valid.to_csv(output_csv_path, index=False, encoding="utf-8")
print(f"Predições detalhadas salvas com sucesso em:\n{output_csv_path.resolve()}")